# Jour 1 · Baseline, prédictions et métriques

## Objectifs

- entraîner une baseline et un premier classifieur
- lire une matrice de confusion
- interpréter précision, rappel, F1 et seuil de décision


## Toujours commencer par une baseline

Une **baseline** est une règle volontairement simple qui fixe le niveau minimum à battre. Ici, `DummyClassifier` prédit selon une stratégie naïve. Si un modèle complexe ne fait pas mieux, sa complexité n'est pas justifiée.


Une baseline peut toujours prédire la classe majoritaire, tirer selon la fréquence des classes ou appliquer une petite règle métier. Elle ne cherche pas à être excellente : elle donne un point de comparaison compréhensible.

Dans le code, `DummyClassifier(strategy="most_frequent")` répond toujours avec la classe la plus fréquente. La régression logistique doit apporter une information utile au-delà de cette stratégie naïve.


![Étapes fit et predict](../ressources/illustrations/jour_01/03_fit_predict.png)

*fit apprend sur le train ; predict utilise ensuite les paramètres appris sur de nouvelles lignes.*


### Préparer les outils, puis ouvrir le tableau

`import` rend disponible une bibliothèque, c'est-à-dire un ensemble d'outils déjà écrits. `as pd`, par exemple, lui donne un nom court. **pandas** lit et manipule les tableaux ; **NumPy** calcule avec des tableaux de nombres ; **Matplotlib** dessine. Une **variable** est un nom associé à une valeur : `df` désignera notre tableau.

Exécutez les cellules de haut en bas, avec **Maj + Entrée**. Une cellule terminée peut produire un texte, un tableau ou un graphique ; l'absence de sortie n'est pas une erreur.


In [ ]:
from pathlib import Path  # Path décrit un chemin de fichier, sur Windows comme sur Linux.
import numpy as np  # np désigne les outils de calcul numérique.
import pandas as pd  # pd désigne les outils de tableaux.
import matplotlib.pyplot as plt  # plt désigne les outils de dessin.
from IPython.display import display  # display affiche joliment un tableau dans Jupyter.

plt.style.use("seaborn-v0_8-whitegrid")  # Ajoute une grille lisible aux graphiques.


Le petit outil suivant retrouve le dossier du cours. `def` définit une **fonction**, un bloc réutilisable ; `for` essaie plusieurs dossiers ; `if` teste une condition ; `return` fournit le résultat. `Path.cwd()` est le dossier courant et `.parents` ses dossiers parents. Le `/` entre chemins assemble les noms, il ne fait pas une division.

C'est une étape technique de chargement, pas une méthode de Machine Learning. Vous pouvez exécuter cette fonction telle quelle.


In [ ]:
def find_course_root():
    """Chercher le dossier qui contient les données du cours."""
    current = Path.cwd()  # Dossier depuis lequel Jupyter exécute ce notebook.
    for candidate in (current, *current.parents):  # * ajoute les parents à la liste d'essais.
        data_path = candidate / "ressources" / "donnees" / "maintenance_machines.csv"
        if data_path.exists():  # Arrêter la recherche dès que le fichier est trouvé.
            return candidate
    raise FileNotFoundError("Ouvrez le notebook depuis le dossier fondamentaux_ml_dl.")

ROOT = find_course_root()  # ROOT garde le chemin du dossier contenant les données.
DATA_FILE = ROOT / "ressources" / "donnees" / "maintenance_machines.csv"


In [ ]:
df = pd.read_csv(DATA_FILE)  # Lire le CSV : df est maintenant un DataFrame (tableau).
display(df.head(3))  # Montrer 3 lignes seulement ; le tableau complet n'est pas modifié.


### Retrouver les entrées et la réponse

Une liste Python utilise des crochets : `["a", "b"]`. `df[features]` sélectionne plusieurs colonnes et garde un tableau 2D ; `df["failure_7d"]` sélectionne une colonne et produit une **Series**, une suite 1D de réponses. Les lignes de `X` et `y` doivent décrire exactement les mêmes observations, dans le même ordre.


In [ ]:
features = [  # Noms des mesures disponibles à l'instant de la décision.
    "temperature_c", "vibration_mm_s", "pressure_bar",
    "load_pct", "age_years", "days_since_maintenance",
]
X = df[features]  # Entrées : une ligne par observation, 6 colonnes par ligne.
y = df["failure_7d"]  # Réponses connues : 0 = pas de panne ; 1 = panne dans 7 jours.
print("X :", X.shape, "| y :", y.shape)  # shape donne les dimensions, pas les valeurs.


### Découper ensemble les entrées et les réponses

`train_test_split` renvoie quatre objets dans cet ordre : entrées conservées, entrées réservées, réponses conservées, réponses réservées. On écrit quatre noms à gauche de `=` pour les récupérer.

`test_size=0.40` réserve 40 % du total dans un bloc **temporaire**, pas dans le test final. `stratify=y` conserve approximativement la proportion de pannes. `random_state=42` refait le même tirage, pour que chacun puisse comparer ses résultats.


In [ ]:
from sklearn.model_selection import train_test_split  # Outil pour séparer des lignes.

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.40,  # Réserver 40 % ; les 60 % restants servent à apprendre.
    stratify=y,  # Conserver approximativement le taux de panne dans les deux groupes.
    random_state=42,  # Fixer le tirage pseudo-aléatoire, sans améliorer le modèle.
)


Le bloc temporaire contient 40 % du total. Le partager en deux moitiés donne **20 % de validation et 20 % de test**. Le train sert à apprendre, la validation à comparer les choix ; on réserve le test pour la décision finale.


In [ ]:
X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp, y_temp,  # Découper uniquement le bloc réservé précédemment.
    test_size=0.50,  # La moitié de 40 % donne 20 % du total pour le test final.
    stratify=y_temp,  # Préserver la proportion de pannes à l'intérieur de ce bloc.
    random_state=42,
)
print("Train / validation / test :", len(X_train), len(X_validation), len(X_test))


**Vérification.** Les trois nombres doivent s'additionner au nombre total de lignes. Pour ces données pédagogiques, chaque ligne concerne une machine différente. Avec plusieurs observations d'une même machine, ou pour prévoir le futur réel, il faudrait aussi séparer les machines ou le temps : une stratification seule ne suffit pas.


### Créer deux points de comparaison

La **baseline** prédit toujours la classe la plus fréquente. Notre modèle combine les mesures, après standardisation. On entraîne les deux sur le même train et on les observe sur la **validation**. Le test reste réservé à la fin ; aucun choix de seuil ne dépend de lui.


In [ ]:
from sklearn.dummy import DummyClassifier  # Modèle naïf, utile comme repère minimum.
from sklearn.pipeline import make_pipeline  # Associer préparation et modèle.
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score

baseline = DummyClassifier(strategy="most_frequent")  # Toujours répondre la classe majoritaire.
baseline.fit(X_train, y_train)  # Identifier cette classe uniquement avec les réponses du train.
baseline_pred = baseline.predict(X_validation)  # Une réponse par ligne de validation.


In [ ]:
model = make_pipeline(
    StandardScaler(),  # Apprendre les unités communes sur le train, puis les réutiliser.
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
)
model.fit(X_train, y_train)  # Ajuster les paramètres à partir du train uniquement.
pred = model.predict(X_validation)  # Décisions 0/1 sur les observations de validation.
print("Baseline accuracy :", round(accuracy_score(y_validation, baseline_pred), 3))
print("Modèle accuracy :", round(accuracy_score(y_validation, pred), 3))


## Pourquoi l'accuracy peut tromper

Avec peu de pannes, prédire toujours « normal » produit une accuracy élevée. Il faut regarder les quatre issues : vrai positif, faux positif, vrai négatif et faux négatif.


![Exemple d’accuracy élevée mais rappel nul](../ressources/illustrations/jour_01/03_accuracy_trompeuse.png)

L’**accuracy** est la proportion totale de bonnes réponses. Elle traite toutefois toutes les lignes de la même façon. Si 94 machines sur 100 ne tombent pas en panne, répondre toujours « normal » donne 94 % d’accuracy tout en manquant les six pannes.

Une métrique doit donc être reliée à la question métier. Pour une classe rare, on regarde séparément la capacité à détecter les événements et la quantité de fausses alertes produites.


![Matrice de confusion en quatre cases](../ressources/illustrations/jour_01/03_matrice_confusion.png)

*La matrice de confusion distingue les bonnes décisions des deux types d’erreurs.*


### Les quatre cases, en langage métier

- **Vrai positif** : une panne est annoncée et elle survient.
- **Faux positif** : une alerte est produite mais aucune panne ne survient.
- **Vrai négatif** : la machine reste normale et le modèle n’alerte pas.
- **Faux négatif** : aucune alerte n’est produite alors qu’une panne survient.

Le mot *positif* signifie ici « le modèle ou la réalité indique la classe panne ». Il ne signifie pas « bon résultat ».

Les deux erreurs n’ont pas nécessairement le même coût. Un faux positif peut déclencher une inspection inutile ; un faux négatif peut laisser survenir une panne non anticipée. Cette asymétrie explique pourquoi le choix d’une métrique et d’un seuil est une décision métier.


### Lire les axes avant les nombres

`confusion_matrix` reçoit d'abord la réalité, ensuite la prédiction. Les **lignes** représentent les réponses réelles, les **colonnes** les réponses prédites. `labels=[0, 1]` impose l'ordre normal puis panne. Une valeur sur la diagonale est une bonne réponse ; les deux autres cases sont les erreurs.


In [ ]:
cm = confusion_matrix(y_validation, pred, labels=[0, 1])  # Compter les quatre issues.
cm_df = pd.DataFrame(
    cm,
    index=["réel normal (0)", "réel panne (1)"],  # Lire la réalité verticalement.
    columns=["prédit normal (0)", "prédit panne (1)"],  # Lire la décision horizontalement.
)
display(cm_df)


## Précision, rappel et F1

La **précision** répond : parmi toutes les alertes produites, quelle proportion correspondait réellement à une panne ? Le **rappel** répond : parmi toutes les pannes réelles, quelle proportion a été détectée ? Le **F1** combine précision et rappel et devient élevé seulement si les deux restent convenables.


![Questions posées par la précision et le rappel](../ressources/illustrations/jour_01/03_precision_rappel.png)

*Une équipe qui ne tolère pas les pannes manquées privilégiera souvent davantage le rappel.*


### Un exemple chiffré

Supposons que le modèle produise 20 alertes et que 8 correspondent réellement à une panne. Sa précision vaut `8 / 20`, soit 40 %. S’il existait 10 pannes au total et qu’il en a trouvé 8, son rappel vaut `8 / 10`, soit 80 %.

Le **score F1** résume précision et rappel par une moyenne qui pénalise fortement une valeur très faible. Pour 40 % de précision et 80 % de rappel, il vaut `2 × 0,40 × 0,80 / (0,40 + 0,80) ≈ 0,533`, soit environ 53 %. Le calcul utilise les proportions 0,40 et 0,80, pas les entiers 40 et 80. Il est utile pour comparer, mais il masque le nombre réel d’alertes et ne connaît pas le coût métier. On conserve donc toujours les métriques détaillées.


### À vous de jouer — calculer avant la bibliothèque

Dans l'image chiffrée : **8 pannes détectées**, **12 fausses alertes**, **2 pannes manquées**. Complétez les deux divisions pour retrouver la précision et le rappel. La précision regarde les **alertes** ; le rappel regarde les **pannes réelles**.


In [ ]:
precision_exemple = None  # À compléter : 8 / (8 + 12).
rappel_exemple = None  # À compléter : 8 / (8 + 2).
print("Précision :", precision_exemple, "| rappel :", rappel_exemple)


### Calculer les métriques de validation

Chaque fonction reçoit les mêmes réponses réelles et prédites. `zero_division=0` renvoie 0 lorsqu'aucune alerte n'est produite au lieu de laisser une division impossible. `pred.sum()` compte les alertes parce que les décisions valent 0 ou 1 ; `int` affiche ce compte comme un entier.


In [ ]:
metrics = pd.Series({  # Une Series relie chaque nom de métrique à sa valeur.
    "accuracy": accuracy_score(y_validation, pred),
    "precision": precision_score(y_validation, pred, zero_division=0),
    "recall": recall_score(y_validation, pred, zero_division=0),
    "f1": f1_score(y_validation, pred, zero_division=0),
    "alertes": int(pred.sum()),  # Nombre d'inspections que déclencheraient ces décisions.
})
display(metrics.round(3))  # Arrondir seulement la présentation des valeurs.


## Une probabilité devient une décision grâce à un seuil

`predict_proba` produit ici un score entre 0 et 1. Par défaut, un seuil proche de 0,5 transforme ce score en classe. Baisser le seuil crée généralement davantage d'alertes et augmente le rappel, mais peut réduire la précision.


![Effet du seuil de décision](../ressources/illustrations/jour_01/03_seuil_decision.png)

Dans `predict_proba(X_validation)[:, 1]`, `[:, 1]` sélectionne pour chaque ligne le score associé à la classe `1`, ici la panne. Un score de 0,70 signifie que le modèle favorise fortement cette classe selon ce qu’il a appris ; ce n’est pas une promesse individuelle de panne.

Si le seuil baisse, davantage de machines passent du côté « alerte » : on trouve souvent plus de pannes, mais on crée aussi davantage de faux positifs. S’il monte, le nombre d’alertes diminue et certaines pannes peuvent être manquées.

Nous comparons effectivement les seuils sur la **validation**, puis nous figeons notre choix avant d'ouvrir le test. Le test ne sert pas à améliorer la règle après avoir vu son score. Avec `class_weight="balanced"`, les scores peuvent ne pas correspondre directement à la fréquence réelle des pannes : leur calibration se vérifierait sur des données indépendantes.


### Voir le tableau des deux scores avant de le découper

`predict_proba` renvoie **deux colonnes** : score de la classe 0, puis score de la classe 1. Pour `[0.30, 0.70]`, le score panne est 0,70. Dans `[:, 1]`, `:` garde toutes les lignes et `1` garde la **deuxième** colonne, puisque Python commence à 0.


In [ ]:
scores_validation = model.predict_proba(X_validation)  # Deux scores pour chaque observation.
print("Ordre des classes :", model.classes_)  # Vérifier que les colonnes correspondent à [0, 1].
print("Premières paires :", scores_validation[:3])  # Montrer trois observations seulement.
proba = scores_validation[:, 1]  # Garder le score panne (classe 1) de chaque ligne.


### Transformer un score en alerte

`proba >= seuil` compare **chaque score** au seuil et renvoie `True` (vrai) ou `False` (faux). `.astype(int)` transforme ces valeurs en **1** et **0**. Cela change la décision sans réentraîner le modèle.


In [ ]:
scores_exemple = np.array([0.20, 0.40, 0.80])  # Un petit tableau de trois scores.
alertes_exemple = (scores_exemple >= 0.50).astype(int)  # Seul 0,80 dépasse le seuil.
print("À 0,50 :", alertes_exemple)  # Résultat attendu : [0, 0, 1].


## À vous de jouer — comparer trois seuils sur la validation

La boucle teste 0,30, 0,50 puis 0,70. Complétez la ligne `flags = None` avec la comparaison vue juste avant. `rows.append(...)` ajoute une ligne de résultats à la liste ; le DataFrame la rend lisible.

Observez : quand le seuil baisse, combien d'alertes sont ajoutées ? Le rappel monte-t-il ? Le F1 choisit ici un compromis pédagogique ; le métier pourrait préférer une autre règle.


In [ ]:
rows = []  # Liste qui recevra une ligne par seuil.
for threshold in [0.30, 0.50, 0.70]:  # Répéter le même calcul pour trois décisions possibles.
    flags = None  # À compléter : (proba >= threshold).astype(int).
    if flags is None:  # Attendre votre ligne avant de calculer les métriques.
        continue
    rows.append({
        "seuil": threshold,
        "alertes": int(flags.sum()),
        "precision": precision_score(y_validation, flags, zero_division=0),
        "recall": recall_score(y_validation, flags, zero_division=0),
        "f1": f1_score(y_validation, flags, zero_division=0),
    })
seuils_validation = pd.DataFrame(rows) if rows else None
if seuils_validation is not None:
    display(seuils_validation.set_index("seuil").round(3))
else:
    print("Complétez flags pour comparer les seuils.")


### Figer une décision avant le contrôle final

Choisissez un des trois seuils en utilisant **uniquement le tableau de validation**, puis renseignez `seuil_retenu`. La prochaine cellule applique le même modèle déjà entraîné au test.

On ne réentraîne pas ici : le seuil a été comparé pour ce modèle précis. Le score final décrit cette règle complète. Le voir puis modifier le seuil reviendrait à transformer le test en validation.


In [ ]:
seuil_retenu = None  # À remplacer par 0.30, 0.50 ou 0.70 après la comparaison de validation.
raison_choix = ""  # Quelle métrique ou contrainte métier justifie votre décision ?
print("Seuil retenu :", seuil_retenu, "| raison :", raison_choix or "à compléter")


In [ ]:
if seuil_retenu is None:
    print("Le test reste fermé : terminez et justifiez votre choix sur la validation.")
else:
    scores_test = model.predict_proba(X_test)[:, 1]  # Appliquer le modèle inchangé au test.
    decisions_test = (scores_test >= seuil_retenu).astype(int)  # Appliquer le seuil figé.
    print("Rappel test :", round(recall_score(y_test, decisions_test, zero_division=0), 3))
    print("Précision test :", round(precision_score(y_test, decisions_test, zero_division=0), 3))
    print("F1 test :", round(f1_score(y_test, decisions_test, zero_division=0), 3))


**À retenir :** le seuil encode un compromis opérationnel ; il n'existe pas de valeur universellement correcte.


Pour approfondir après la séance : [documentation scikit-learn sur le seuil de décision](https://scikit-learn.org/stable/modules/classification_threshold.html). Elle distingue le score produit par le modèle et l'action décidée à partir de ce score.
